##CREATE CATALOG

- globalmart_dev is the single catalog for the entire POC.
- Bronze, Silver, and Gold all live as schemas inside this catalog.
- Using one catalog keeps lineage graphs connected across all three layers —
- Unity Catalog can trace a Gold metric back through Silver to Bronze to the
- raw Volume file without crossing catalog boundaries.


In [0]:
spark.sql("""
    CREATE CATALOG IF NOT EXISTS globalmart_dev
    COMMENT 'GlobalMart Data Intelligence Platform — POC catalog.
             Contains Bronze (raw), Silver (cleansed), and Gold (star schema)
             schemas. All data originates from 6 regional retail systems.'
""")

print("Catalog globalmart_dev: ready")


##USE CATALOG
- Set the default catalog for all subsequent SQL statements in this session.
- Avoids having to fully qualify every object as globalmart_dev.<schema>.<table>
- in the setup script (pipelines still use fully qualified names).

In [0]:
spark.sql("USE CATALOG globalmart_dev")
print("Active catalog: globalmart_dev")


#CREATE BRONZE SCHEMA
- Bronze schema holds raw ingestion tables only — one per entity.
- No Silver or Gold tables ever land here.
-
- The schema comment is important: it appears in the Unity Catalog UI and
- tells anyone browsing the metastore exactly what this schema contains
- and what the data quality expectations are.

In [0]:
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS globalmart_dev.bronze
    COMMENT 'Bronze layer — raw ingestion from all 6 regional source systems.
             Tables: customers, orders, transactions, returns, products, vendors.
             Data is loaded exactly as received — no transformations applied.
             Every record carries _source_file_path, _source_file_name,
             _source_modified_time, _ingested_at, and source_region columns
             for full audit traceability back to the originating file.'
""")

print("Schema globalmart_dev.bronze: ready")


# =============================================================================
# CELL 4 — CONFIRM VOLUME IS ACCESSIBLE
# =============================================================================
# The volume was created when you uploaded the raw files in the Databricks UI.
# This cell confirms the volume path resolves correctly and lists what is
# inside so you can verify all 16 files are present before running the pipeline.


In [0]:
VOLUME_PATH = "/Volumes/globalmart_dev/default/globalmart_retail_data"

files = dbutils.fs.ls(f"{VOLUME_PATH}/GlobalMart_Retail_Data")
print(f"Root-level entries in volume ({len(files)} items):")
for f in files:
    print(f"  {f.path}")

# Recurse into each Region sub-folder to confirm regional files
print("\nRegional files:")
for item in files:
    if "Region" in item.name:
        region_files = dbutils.fs.ls(item.path)
        for rf in region_files:
            print(f"  {rf.path}")


# The volume was created when you uploaded the raw files in the Databricks UI.
# This cell confirms the volume path resolves correctly and lists what is
# inside so you can verify all 16 files are present before running the pipeline.
#
# Expected output:
#   GlobalMart_Retail_Data/          (sub-directory)
#   GlobalMart_Retail_Data/transactions_3.csv
#   GlobalMart_Retail_Data/vendors.csv
#   GlobalMart_Retail_Data/returns_2.json
#   GlobalMart_Retail_Data/products.json
#   GlobalMart_Retail_Data/Region 1/customers_1.csv
#   GlobalMart_Retail_Data/Region 1/orders_1.csv
#   ... etc for all regions

In [0]:
def walk_volume(path):
    """Recursively list all files under a volume path."""
    all_files = []
    items = dbutils.fs.ls(path)
    for item in items:
        if item.isDir():
            all_files.extend(walk_volume(item.path))
        else:
            all_files.append(item.path)
    return all_files

all_files = walk_volume(f"{VOLUME_PATH}/GlobalMart_Retail_Data")

# Group by entity
entity_counts = {
    "customers": [f for f in all_files if "customers_" in f],
    "orders":    [f for f in all_files if "orders_" in f],
    "transactions": [f for f in all_files if "transactions_" in f],
    "returns":   [f for f in all_files if "returns_" in f],
    "products":  [f for f in all_files if "products" in f],
    "vendors":   [f for f in all_files if "vendors" in f],
}

print(f"Total files found: {len(all_files)}\n")
for entity, paths in entity_counts.items():
    status = "OK" if len(paths) > 0 else "MISSING"
    print(f"  [{status}] {entity}: {len(paths)} file(s)")
    for p in paths:
        print(f"         {p}")

expected_total = 16
if len(all_files) == expected_total:
    print(f"\nAll {expected_total} source files confirmed. Safe to run 02_bronze_pipeline.py")
else:
    print(f"\nWARNING: Expected {expected_total} files, found {len(all_files)}.")
    print("Resolve missing files before running the Bronze pipeline.")


# =============================================================================
# CELL 6 — RBAC: ENGINEERING ACCESS ON BRONZE SCHEMA
# =============================================================================
- Engineering group needs full privileges on the bronze schema to:
   - Run and monitor the DLT pipeline
   - Query Bronze tables directly for debugging
   - Create and drop tables during pipeline development
-
- USE SCHEMA is required in addition to SELECT so engineers can browse
- the schema in the Unity Catalog UI and see table metadata.
-
- Note: Finance, Returns, and Merch team grants are NOT set here.
- Those teams access Gold layer only — their grants are added in the
- Gold setup script (05_uc_gold_grants.py, to be created later).
- Setting Gold grants now would be premature since Gold tables don't exist yet.


In [0]:
# Create the engineering group if it doesn't exist
try:
    spark.sql("CREATE GROUP globalmart_engineering")
    print("Created group: globalmart_engineering")
except Exception as e:
    if "PRINCIPAL_ALREADY_EXISTS" in str(e):
        print("Group globalmart_engineering already exists")
    else:
        raise

spark.sql("""
    GRANT USE SCHEMA ON SCHEMA globalmart_dev.bronze
    TO `globalmart_engineering`
""")

spark.sql("""
    GRANT SELECT ON SCHEMA globalmart_dev.bronze
    TO `globalmart_engineering`
""")

spark.sql("""
    GRANT MODIFY ON SCHEMA globalmart_dev.bronze
    TO `globalmart_engineering`
""")

spark.sql("""
    GRANT CREATE TABLE ON SCHEMA globalmart_dev.bronze
    TO `globalmart_engineering`
""")

print("RBAC grants on globalmart_dev.bronze: applied for globalmart_engineering")

# CELL 7 — RBAC: VOLUME ACCESS FOR PIPELINE SERVICE PRINCIPAL

- The DLT pipeline runs under a service principal (or your user account in
- the POC). That principal needs READ VOLUME on the raw file volume so Auto
- Loader can scan and ingest files.
-
- WRITE VOLUME is also granted so Auto Loader can write schema location and
- checkpoint state back into the volume (_autoloader_schema / _autoloader_checkpoint
- sub-directories created automatically on first pipeline run).
-
- Replace 'globalmart_engineering' with your specific service principal name
- if running the pipeline under a dedicated SP rather than a group.


In [0]:

spark.sql("""
    GRANT READ VOLUME ON VOLUME globalmart_dev.default.globalmart_retail_data
    TO `globalmart_engineering`
""")

spark.sql("""
    GRANT WRITE VOLUME ON VOLUME globalmart_dev.default.globalmart_retail_data
    TO `globalmart_engineering`
""")

print("Volume grants: globalmart_engineering can read and write the raw data volume")


#VERIFY SETUP BEFORE RUNNING PIPELINE

- Final check: confirm catalog and schema exist and are queryable.
- Also lists any tables already in the bronze schema (should be empty on
- first run, populated after 02_bronze_pipeline.py runs successfully).

In [0]:
# =============================================================================


print("=== SETUP VERIFICATION ===\n")

# Confirm catalog exists
catalogs = spark.sql("SHOW CATALOGS").collect()
catalog_names = [r[0] for r in catalogs]
print(f"Catalog globalmart_dev exists: {'globalmart_dev' in catalog_names}")

# Confirm bronze schema exists
schemas = spark.sql("SHOW SCHEMAS IN globalmart_dev").collect()
schema_names = [r[0] for r in schemas]
print(f"Schema bronze exists: {'bronze' in schema_names}")

# List tables in bronze (empty before pipeline runs)
tables = spark.sql("SHOW TABLES IN globalmart_dev.bronze").collect()
if tables:
    print(f"\nTables in globalmart_dev.bronze ({len(tables)}):")
    for t in tables:
        print(f"  {t.tableName}")
else:
    print("\nNo tables in globalmart_dev.bronze yet — expected before pipeline run")

print("\nCatelog Setup complete")


### Creating silver schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS silver